<a href="https://colab.research.google.com/github/JulTob/Modelling/blob/main/GLM/Explorator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Descarga Alternativa y Estable de Datos de Fórmula 1

Usaremos un repositorio de datos de F1 sumamente estable y público para garantizar la correcta obtención de los archivos `results.csv` y `drivers.csv`.

In [14]:
# URLs del repositorio histórico consolidado de Ergast F1 (base de datos abierta oficial)
url_results <- "https://raw.githubusercontent.com/thef1dev/f1_datasource/main/ergast_parser/results.csv"
url_drivers <- "https://raw.githubusercontent.com/thef1dev/f1_datasource/main/ergast_parser/drivers.csv"


[1] "¡Archivos de Fórmula 1 (TidyTuesday Mirror Oficial) descargados con éxito!"


### Carga, Integración y Modelado GLM

Una vez descargados, cargamos ambos archivos, realizamos la unión por `driverId`, definimos nuestra variable objetivo `podium` (si finalizó en el top 3) y ajustamos el modelo binomial para predecir el éxito basándonos en la posición de salida (`grid`).

In [19]:
# Carga de los archivos CSV
u <- "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2021/2021-09-07/"

results   <- read.csv(paste0(u,"results.csv"), na.strings="\\N")
drivers <- read.csv("drivers.csv")
races <- read.csv(paste0(u,"races.csv"),   na.strings="\\N")
st    <- read.csv(paste0(u,"status.csv"))
f1 <- merge(merge(results, races[,c("raceId","year","round","circuitId")], by="raceId"), st, by="statusId")
f1 <- subset(f1, year >= 2000 & grid > 0)
f1$podio <- as.integer(f1$positionOrder <= 3)

# Unión de los sets de datos
f1_data <- merge(results, drivers, by = "driverId")

# Creación de la variable dependiente (1 si terminó en podio: posiciones 1, 2 o 3)
f1_data$podium <- ifelse(f1_data$positionOrder <= 3, 1, 0)

# Filtrar registros válidos con posiciones de salida lógicas (grid > 0)
f1_clean <- subset(f1_data, grid > 0 & !is.na(podium))

# Ajuste del modelo de regresión logística lineal generalizada
modelo_f1 <- glm(podium ~ grid, data = f1_clean, family = binomial)

# Mostrar el resumen de los resultados del modelo
summary(modelo_f1)


Call:
glm(formula = podium ~ grid, family = binomial, data = f1_clean)

Coefficients:
             Estimate Std. Error z value Pr(>|z|)    
(Intercept)  0.713700   0.039303   18.16   <2e-16 ***
grid        -0.333146   0.006205  -53.69   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 18614  on 23626  degrees of freedom
Residual deviance: 12677  on 23625  degrees of freedom
AIC: 12681

Number of Fisher Scoring iterations: 7


In [20]:
lm(podio ~ grid, data = f1)


Call:
lm(formula = podio ~ grid, data = f1)

Coefficients:
(Intercept)         grid  
    0.45013     -0.02769  
